# 03 · Reference choice and an audio LLM's emotion judgement (Qwen2.5-Omni pilot)

## Purpose
HW3 showed that **which reference loudness is measured against** changes what a small speech-emotion model is sensitive to:
a global reference (A) is sensitive to recording volume, a within-clip reference (B) removes that, and a personal reference (D) helps accuracy but needs enrollment clips.
This notebook asks the same question of an audio LLM: give Qwen2.5-Omni-7B the clip plus short acoustic descriptions computed against different references, and see how its judgement moves.
It also runs a counterfactual: keep the audio fixed and change only the written loudness level, to test whether the model follows the text evidence or its own hearing.

## Detailed outline
1. **Environment and paths:** cache locations on scratch, GPU check.
2. **Configuration:** model ID, data paths, number of clips, run name, execution switches.
3. **Install (once):** Python packages for the model.
4. **Audio download (once):** CREMA-D WAV files; features and votes come from the HW3 repository.
5. **Cues, buckets and enrollment:** clip-level loudness/pitch cues, tertile buckets from training speakers, personal reference from neutral enrollment clips.
6. **Prompts and dry run:** inspect the exact text for each condition before loading the model.
7. **Load the model and test one clip.**
8. **Run all clips (resumable):** one forward pass per clip × condition; letter probabilities saved per row.
9. **Summary and inspection:** accuracy, listener agreement, neutral→sad, loudness association, counterfactual effect.

## Conditions
| Condition | Text given with the audio | HW3 analogue |
|---|---|---|
| `audio_only` | none | control |
| `global` | loudness and pitch **level vs other speakers** + within-clip variation | A |
| `within` | within-clip variation only | B |
| `personal` | loudness and pitch **level vs this speaker's own neutral voice** + within-clip variation | D |
| `cf_loud_low` / `cf_loud_high` | `global` text with the loudness level forced to low / high; audio unchanged | evidence test |

## Requirements and execution (UVA HPC)
Start JupyterLab through Open OnDemand with **one A100 GPU** (40 GB is enough; the model in bf16 needs roughly 20–30 GB).
Run the notebook from the root of the HW3 repository (the folder that contains `training/` and `data/`).
The first model load downloads about 15–20 GB of weights; cache them on scratch, not in your home directory (cell 1).
Execution switches are disabled by default. Review the configuration, then turn on the `RUN_*` switches one at a time.
No results are prepopulated.

## 1 · Environment, cache and GPU
Set the cache directory **before** importing `transformers`. `CACHE_DIR` defaults to `/scratch/$USER/hf_cache`; change it if your scratch path differs.
If no GPU is listed, restart the session with a GPU partition selected.

In [ ]:
import os
from pathlib import Path

USER = os.environ.get("USER", "user")
CACHE_DIR = Path(os.environ.get("SCRATCH", f"/scratch/{USER}")) / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)          # model weights go here, not to $HOME

PROJECT_DIR = Path.cwd()                          # must be the HW3 repository root
assert (PROJECT_DIR / "training" / "train.py").exists(), "Start Jupyter in the repository root (it must contain training/)."

import csv, hashlib, json, sys, time
from collections import Counter
from datetime import datetime, timezone
import numpy as np
sys.path.insert(0, str(PROJECT_DIR))
from training.train import speaker_split

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), f"| memory {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
except ImportError:
    print("torch is not installed yet; run section 3.")
print("Project:", PROJECT_DIR)
print("HF cache:", CACHE_DIR)

## 2 · Configuration
`LIMIT=200` runs a fixed random subset of the 1,421 test queries (enough to see whether the conditions differ; about 10–20 minutes on one A100).
Set `LIMIT=0` for all test queries (about 1 hour). The loudness association (section 9) needs the full set: a 200-clip subset has only about 20 neutral clips.
Use a new `RUN_NAME` whenever you change the model, the prompt or the conditions; the run refuses to mix settings in one output folder.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Omni-7B"
FEATURES = PROJECT_DIR / "data" / "features_K8.npz"
VOTES = PROJECT_DIR / "data" / "tabulatedVotes.csv"
AUDIO_DIR = PROJECT_DIR / "data" / "crema"

LIMIT = 200                       # 0 = all 1,421 test queries
RUN_NAME = "omni7b_pilot200_v2"   # v2: symmetric personal-reference wording
OUT_DIR = PROJECT_DIR / "hw3" / "omni_pilot" / RUN_NAME

CONDITIONS = ["audio_only", "global", "within", "personal", "cf_loud_low", "cf_loud_high"]
ENROLL = ["DFA", "TAI", "IOM"]    # neutral clips of these sentences = personal reference; never evaluated
SEED = 20261107

RUN_INSTALL = False
RUN_DOWNLOAD_AUDIO = False
RUN_DRY = True
RUN_LOAD_MODEL = False
RUN_ALL = False

for p in (FEATURES, VOTES):
    print(p.name, "found" if p.exists() else "MISSING")
print("WAV files:", len(list(AUDIO_DIR.glob("*.wav"))) if AUDIO_DIR.exists() else 0, "of 7,442")

## 3 · Install packages (once)
Qwen2.5-Omni needs `transformers` 4.52 or newer. After installing, **restart the kernel** and rerun from section 1.

In [ ]:
if RUN_INSTALL:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-U",
                    "transformers>=4.52", "accelerate", "librosa", "soundfile", "scipy"], check=True)
else:
    print("Install disabled. Set RUN_INSTALL=True to install, then restart the kernel.")
try:
    import transformers
    print("transformers", transformers.__version__)
except ImportError:
    print("transformers is not installed.")

## 4 · Download CREMA-D audio (once, about 600 MB)
The GitHub package contains features and listener votes but not the WAV files. This calls the repository's downloader, which skips files that already exist.
If the compute node has no internet access, run `python scripts/download_crema.py` on a login node instead.

In [ ]:
import subprocess
if RUN_DOWNLOAD_AUDIO:
    subprocess.run([sys.executable, "scripts/download_crema.py", "--workers", "16"], check=True, cwd=PROJECT_DIR)
else:
    print("Audio download disabled.")
print("WAV files:", len(list(AUDIO_DIR.glob("*.wav"))) if AUDIO_DIR.exists() else 0, "of 7,442")

## 5 · Cues, buckets and the personal reference
**Inputs:** the HW2 feature cache (7,442 clips × 8 segments × 88 features). **Outputs:** one low/mid/high description per cue.

- Loudness level = mean log-RMS over the 8 segments; pitch level = mean log-F0; loudness variation = SD of log-RMS across segments; pitch variability = mean within-segment log-F0 SD.
- Cut points are tertiles of **training speakers' non-enrollment clips only**.
- Personal reference = mean of the speaker's three neutral enrollment clips; the personal description buckets the difference from it, with cut points again from training speakers.
- The speaker split is the HW2 split (18 test speakers). Enrollment clips are removed from evaluation.

In [ ]:
EMOTIONS = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]
WORDS = ["anger", "disgust", "fear", "happiness", "neutral", "sadness"]
LETTERS = ["A", "B", "C", "D", "E", "F"]
IDX = {"logf0_mean": 80, "logf0_std": 81, "logrms_mean": 84}

def clip_cues(X):
    return {"loud_level": X[:, :, IDX["logrms_mean"]].mean(1),
            "pitch_level": X[:, :, IDX["logf0_mean"]].mean(1),
            "loud_var": X[:, :, IDX["logrms_mean"]].std(1),
            "pitch_var": X[:, :, IDX["logf0_std"]].mean(1)}

def tertiles(v):
    return np.quantile(v, [1 / 3, 2 / 3])

def bucket(v, cuts, words=("low", "mid", "high")):
    return words[int(np.searchsorted(cuts, v))]

REL_WORDS = ("lower", "about the same", "higher")

def prepare(features):
    d = np.load(features)
    X, spk, sent, emo = d["X"], d["speaker"], d["sentence"], d["emotion"]
    groups = speaker_split(spk)
    enroll = np.isin(sent, ENROLL) & (emo == "NEU")
    train = np.isin(spk, groups["train"]) & ~enroll
    test = np.flatnonzero(np.isin(spk, groups["test"]) & ~enroll)
    cues = clip_cues(X)
    ref = {s: {k: cues[k][enroll & (spk == s)].mean() for k in ("loud_level", "pitch_level")} for s in np.unique(spk)}
    rel = {k: np.array([cues[k][i] - ref[spk[i]][k] for i in range(len(spk))]) for k in ("loud_level", "pitch_level")}
    cuts = {k: tertiles(cues[k][train]) for k in cues}
    # personal reference: symmetric band around the speaker's own neutral level.
    # "about the same" = within the 75th percentile of |deviation| of training speakers' NEUTRAL query clips,
    # so typical neutral speech is described as about the same as the speaker's neutral voice (v1 used tertiles
    # of all emotions, which called 13 of 25 neutral clips "low" in pitch).
    neu_train = train & (emo == "NEU")
    for k in rel:
        delta = float(np.quantile(np.abs(rel[k][neu_train]), 0.75))
        cuts[f"rel_{k}"] = np.array([-delta, delta])
    assert len(test) == 1421 and not np.any(enroll[test])
    return {"d": d, "test": test, "cues": cues, "rel": rel, "cuts": cuts, "ref": ref, "groups": groups}

P = prepare(FEATURES)
d = P["d"]
order = P["test"].copy()
np.random.default_rng(SEED).shuffle(order)
if LIMIT:
    order = order[:LIMIT]
print("Test queries:", len(P["test"]), "| this run:", len(order))
print("Actor labels in this run:", Counter(d["emotion"][order]))
print("Cut points:", {k: np.round(v, 3).tolist() for k, v in P["cuts"].items()})

## 6 · Prompts and dry run
The model answers a 6-way multiple choice; we read the next-token probabilities of the letters A–F, so each clip × condition is **one forward pass** and always returns a full distribution.
The option order is fixed, so a letter preference affects all conditions equally; compare conditions with each other rather than reading absolute accuracy alone.
Edit the wording here if needed; `PROMPT_VERSION` changes with it, and a changed version needs a new `RUN_NAME`.

In [ ]:
SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving "
          "auditory and visual inputs, as well as generating text and speech.")
QUESTION = ("Listen to the speech clip and judge which emotion the speaker expresses through their voice.\n"
            "{evidence}\nOptions:\n{options}\nAnswer with the letter only.")

def describe(i, condition):
    c, cuts, rel = P["cues"], P["cuts"], P["rel"]
    var = [f"- loudness variation within the clip: {bucket(c['loud_var'][i], cuts['loud_var'])}",
           f"- pitch variability within the clip: {bucket(c['pitch_var'][i], cuts['pitch_var'])}"]
    if condition == "audio_only":
        return ""
    if condition == "within":
        lines = var
    elif condition == "personal":
        lines = [f"- loudness level, compared with this speaker's own neutral voice: {bucket(rel['loud_level'][i], cuts['rel_loud_level'], REL_WORDS)}",
                 f"- pitch level, compared with this speaker's own neutral voice: {bucket(rel['pitch_level'][i], cuts['rel_pitch_level'], REL_WORDS)}"] + var
    else:
        loud = {"cf_loud_low": "low", "cf_loud_high": "high"}.get(condition, bucket(c["loud_level"][i], cuts["loud_level"]))
        lines = [f"- loudness level, compared with other speakers: {loud}",
                 f"- pitch level, compared with other speakers: {bucket(c['pitch_level'][i], cuts['pitch_level'])}"] + var
    return "\nAutomatically measured acoustic cues:\n" + "\n".join(lines) + "\n"

def question(i, condition):
    options = "\n".join(f"{l}. {w}" for l, w in zip(LETTERS, WORDS))
    return QUESTION.format(evidence=describe(i, condition), options=options)

PROMPT_VERSION = hashlib.sha256((SYSTEM + QUESTION + json.dumps(WORDS)).encode()).hexdigest()[:16]
print("PROMPT_VERSION:", PROMPT_VERSION)
neu_in_run = [i for i in order if d["emotion"][i] == "NEU"]
print("Personal wording on the neutral clips of this run:",
      {k: Counter(bucket(P["rel"][k][i], P["cuts"][f"rel_{k}"], REL_WORDS) for i in neu_in_run) for k in ("loud_level", "pitch_level")})
if RUN_DRY:
    i = order[0]
    for c in CONDITIONS:
        print(f"=== {d['fname'][i]} · {c}\n{question(i, c)}\n")

## 7 · Load the model and test one clip
Uses `Qwen2_5OmniThinkerForConditionalGeneration`, which produces text only and does not load the speech generator.
`letter_mass` is the share of the next-token probability that falls on A–F. Values close to 1 mean the model is answering in the requested format; low values mean the prompt needs work before the full run.
If the processor raises a `TypeError` about `audio`, your transformers version names it `audios`; the function handles both.

In [ ]:
model = processor = None
if RUN_LOAD_MODEL:
    import librosa
    from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration
    model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto").eval()
    processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
    LETTER_IDS = []
    for l in LETTERS:
        ids = processor.tokenizer.encode(l, add_special_tokens=False)
        assert len(ids) == 1, f"Letter {l} is not a single token: {ids}"
        LETTER_IDS.append(ids[0])
    print("Loaded", MODEL_ID, "| GPU memory used:", f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")

def letter_probs(wav, text):
    conv = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "audio", "audio": "clip.wav"}, {"type": "text", "text": text}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        inputs = processor(text=prompt, audio=[wav], return_tensors="pt", padding=True)
    except TypeError:
        inputs = processor(text=prompt, audios=[wav], return_tensors="pt", padding=True)
    inputs = {k: (v.to(model.device, model.dtype) if v.is_floating_point() else v.to(model.device)) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    full = torch.softmax(logits, -1)[LETTER_IDS]
    return (full / full.sum()).cpu().numpy(), float(full.sum())

if model is not None:
    i = order[0]
    wav, _ = librosa.load(str(AUDIO_DIR / str(d["fname"][i])), sr=16000, mono=True)
    for c in ("audio_only", "global"):
        p, mass = letter_probs(wav, question(i, c))
        print(d["fname"][i], c, "| actor:", d["emotion"][i], "| letter_mass:", round(mass, 3))
        print("   ", {e: round(float(x), 3) for e, x in zip(EMOTIONS, p)})
else:
    print("Model not loaded. Set RUN_LOAD_MODEL=True after the dry run looks right.")

### 7b · Where does the rest of the probability go? (optional)
`letter_mass` below about 0.9 means the model often wants to start its answer with something other than a bare letter.
This prints the five most likely first tokens for a few clips, so you can see whether the missing mass sits on variants such as " E", "E." or a word like "The".
If most of it sits on letter variants, add those token ids to the scoring; if it sits on words, tighten the instruction.

In [ ]:
if model is not None:
    for i in order[:5]:
        wav, _ = librosa.load(str(AUDIO_DIR / str(d["fname"][i])), sr=16000, mono=True)
        conv = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                {"role": "user", "content": [{"type": "audio", "audio": "clip.wav"}, {"type": "text", "text": question(i, "audio_only")}]}]
        prompt = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        try:
            inputs = processor(text=prompt, audio=[wav], return_tensors="pt", padding=True)
        except TypeError:
            inputs = processor(text=prompt, audios=[wav], return_tensors="pt", padding=True)
        inputs = {k: (v.to(model.device, model.dtype) if v.is_floating_point() else v.to(model.device)) for k, v in inputs.items()}
        with torch.no_grad():
            pr = torch.softmax(model(**inputs).logits[0, -1].float(), -1)
        top = torch.topk(pr, 5)
        print(d["fname"][i], [(repr(processor.tokenizer.decode([int(t)])), round(float(p), 3)) for p, t in zip(top.values, top.indices)])
else:
    print("Load the model first (section 7).")

## 8 · Run all clips (resumable)
- Each finished clip is written and flushed; rerunning skips completed clip × condition rows.
- The manifest stores the model ID, prompt version, conditions, cut points, package versions and GPU. A different configuration in the same `OUT_DIR` raises an error; choose a new `RUN_NAME`.
- Progress prints every 20 clips.

In [ ]:
def run_all():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    settings = {"model": MODEL_ID, "prompt_version": PROMPT_VERSION, "conditions": CONDITIONS, "enroll": ENROLL,
                "limit": LIMIT, "seed": SEED, "cuts": {k: np.round(v, 6).tolist() for k, v in P["cuts"].items()}}
    manifest = OUT_DIR / "manifest.json"
    if manifest.exists() and json.loads(manifest.read_text())["settings"] != settings:
        raise ValueError("Settings changed; choose a new RUN_NAME.")
    if not manifest.exists():
        import transformers
        manifest.write_text(json.dumps({"settings": settings, "system": SYSTEM, "question": QUESTION,
            "transformers": transformers.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0), "started_utc": datetime.now(timezone.utc).isoformat(),
            "clips": [str(d["fname"][i]) for i in order]}, indent=1))
    path = OUT_DIR / "predictions.csv"
    done = {(r["fname"], r["condition"]) for r in csv.DictReader(open(path))} if path.exists() else set()
    new = not path.exists()
    t0 = time.time()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["fname", "speaker", "actor_label", "condition", "pred"] + [f"p_{e}" for e in EMOTIONS] + ["letter_mass"])
        for n, i in enumerate(order, 1):
            fname = str(d["fname"][i])
            todo = [c for c in CONDITIONS if (fname, c) not in done]
            if todo:
                wav, _ = librosa.load(str(AUDIO_DIR / fname), sr=16000, mono=True)
                for c in todo:
                    p, mass = letter_probs(wav, question(i, c))
                    w.writerow([fname, str(d["speaker"][i]), str(d["emotion"][i]), c, EMOTIONS[int(p.argmax())]]
                               + [f"{x:.5f}" for x in p] + [f"{mass:.4f}"])
                f.flush()
            if n % 20 == 0:
                print(f"{n}/{len(order)} clips | {time.time() - t0:.0f} s", flush=True)
    print("Done:", path)

if RUN_ALL:
    assert model is not None, "Load the model first (section 7)."
    run_all()
else:
    print("Full run disabled. Set RUN_ALL=True after the one-clip test looks right.")

## 9 · Summary
Works on partial output: only clips that have all six conditions are compared.

How to read it:
1. **`audio_only` accuracy first.** Near 17% (chance for six classes) means the model barely hears the emotion and the comparisons below say little.
2. **Condition differences:** how much the written cues move accuracy, listener agreement and the share of each predicted emotion; `global` versus `personal` is the reference question.
3. **Counterfactual:** with identical audio, how often the answer changes when only the written loudness switches from low to high, and how P(sadness) and P(anger) shift.
4. **Loudness association:** Spearman ρ across test speakers between enrollment loudness and the share of neutral clips called sad. Listeners give about −0.61 on the full test set; the question is whether a condition amplifies that. Use the full run for this; 18 speakers give wide uncertainty either way.

In [ ]:
from scipy.stats import spearmanr

def summarize():
    rows = list(csv.DictReader(open(OUT_DIR / "predictions.csv")))
    votes = {r["fileName"]: np.array([int(r[c]) for c in "ADFHNS"])
             for r in csv.DictReader(open(VOTES)) if int(r[""]) // 100000 == 1}
    by = {}
    for r in rows:
        by.setdefault(r["condition"], []).append(r)
    have = {k: {r["fname"] for r in v} for k, v in by.items()}
    keep = set.intersection(*[have.get(k, set()) for k in CONDITIONS])
    loud_ref = {s: v["loud_level"] for s, v in P["ref"].items()}
    table = []
    for c in CONDITIONS:
        rr = [r for r in by.get(c, []) if r["fname"] in keep]
        y = np.array([r["actor_label"] for r in rr]); pred = np.array([r["pred"] for r in rr])
        pv = np.array([[float(r[f"p_{e}"]) for e in EMOTIONS] for r in rr])
        vv = np.array([votes[Path(r["fname"]).stem] for r in rr])
        plural = vv == vv.max(1, keepdims=True)
        pidx = np.array([EMOTIONS.index(p) for p in pred])
        spk = np.array([r["speaker"] for r in rr]); neu = y == "NEU"
        per = [(loud_ref[s], np.mean(pred[neu & (spk == s)] == "SAD")) for s in np.unique(spk[neu])]
        table.append({"condition": c, "n": len(rr),
            "acc_vs_actor": np.mean(pred == y),
            "in_listener_plurality": np.mean(plural[np.arange(len(rr)), pidx]),
            "xent_to_votes": np.mean(-(vv / vv.sum(1, keepdims=True) * np.log(pv + 1e-9)).sum(1)),
            "neutral_to_sad": np.mean(pred[neu] == "SAD") if neu.any() else np.nan,
            "rho_loud_vs_neu_sad": spearmanr(*zip(*per)).statistic if len(per) > 3 else np.nan,
            "letter_mass": np.mean([float(r["letter_mass"]) for r in rr]),
            **{f"share_{e}": np.mean(pred == e) for e in EMOTIONS}})
    lo = {r["fname"]: r for r in by.get("cf_loud_low", []) if r["fname"] in keep}
    hi = {r["fname"]: r for r in by.get("cf_loud_high", []) if r["fname"] in keep}
    common = sorted(set(lo) & set(hi))
    cf = {"n": len(common), "top1_changed": np.mean([lo[f]["pred"] != hi[f]["pred"] for f in common]),
          **{f"dP_{e}_high_minus_low": np.mean([float(hi[f][f"p_{e}"]) - float(lo[f][f"p_{e}"]) for f in common]) for e in EMOTIONS}}
    lis = {}
    for i in P["test"]:
        if d["emotion"][i] == "NEU" and str(d["fname"][i]) in keep:
            v = votes[Path(str(d["fname"][i])).stem]
            lis.setdefault(str(d["speaker"][i]), []).append(v[5] / v.sum())
    listener_rho = spearmanr([loud_ref[s] for s in lis], [np.mean(v) for v in lis.values()]).statistic if len(lis) > 3 else np.nan
    out = {"n_clips_complete": len(keep), "conditions": table, "counterfactual_loudness": cf, "listener_rho_loud_vs_sad_votes": listener_rho}
    (OUT_DIR / "summary.json").write_text(json.dumps(out, indent=1, default=float))
    return out

if (OUT_DIR / "predictions.csv").exists():
    S = summarize()
    print("Clips with all conditions:", S["n_clips_complete"])
    cols = ["condition", "n", "acc_vs_actor", "in_listener_plurality", "xent_to_votes", "neutral_to_sad", "rho_loud_vs_neu_sad", "letter_mass"]
    print("  ".join(f"{c:>20s}" for c in cols))
    for r in S["conditions"]:
        print("  ".join(f"{r[c]:>20}" if isinstance(r[c], str) else f"{r[c]:>20.3f}" for c in cols))
    print("\nCounterfactual (same audio, written loudness high vs low):", {k: round(float(v), 3) for k, v in S["counterfactual_loudness"].items()})
    print("Listener ρ (enrollment loudness vs sad votes on neutral clips):", round(float(S["listener_rho_loud_vs_sad_votes"]), 3))
else:
    print("No predictions yet.")

In [ ]:
if (OUT_DIR / "predictions.csv").exists():
    import matplotlib.pyplot as plt
    S = summarize()
    names = [r["condition"] for r in S["conditions"]]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].bar(names, [100 * r["acc_vs_actor"] for r in S["conditions"]], color="#5b8bd6")
    ax[0].axhline(100 / 6, ls=":", color="gray"); ax[0].set_title("Accuracy vs actor label (%)"); ax[0].tick_params(axis="x", rotation=30)
    shares = np.array([[r[f"share_{e}"] for e in EMOTIONS] for r in S["conditions"]])
    left = np.zeros(len(names))
    for j, e in enumerate(EMOTIONS):
        ax[1].barh(names, 100 * shares[:, j], left=left, label=e); left += 100 * shares[:, j]
    ax[1].set_title("Share of predicted emotions (%)"); ax[1].legend(ncol=6, fontsize=7, loc="lower center", bbox_to_anchor=(.5, -.35))
    fig.tight_layout(); fig.savefig(OUT_DIR / "summary.png", dpi=150); plt.show()

## Notes and limits
- CREMA-D is acted English speech with fixed recording conditions; results do not transfer to other settings without testing.
- The cues are coarse (three buckets, four cues) and computed from the HW2 feature cache, not from word-aligned measurements as in ADEPT.
- Letter-probability scoring measures what the model prefers to answer, not a generated explanation; it says nothing about whether a written rationale would be faithful.
- The test speakers were used throughout HW3; treat any pattern found here as exploratory and confirm it on new speakers before claiming it.